# Xenium Data Preprocessing and Quality Control

This notebook handles the initial preprocessing steps for Xenium spatial transcriptomics data:
- Loading data from h5 files
- Quality control metrics
- Filtering low-quality cells and genes
- Normalization
- Highly variable gene selection
- Dimensionality reduction (PCA, UMAP)

**Input:** Raw h5 files from Xenium platform

**Output:** Preprocessed AnnData object ready for downstream analysis

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import h5py
import warnings
warnings.filterwarnings('ignore')

# Set plotting parameters
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=80, facecolor='white', frameon=False)
sns.set_style('whitegrid')

print(f"Scanpy version: {sc.__version__}")
print(f"Squidpy version: {sq.__version__}")

## 1. Define Paths and Parameters

In [ ]:
# Define paths
DATA_DIR = Path("../data/raw")  # Directory containing h5 files
OUTPUT_DIR = Path("../data/processed")
FIGURES_DIR = Path("../figures/01_preprocessing")

# Create output directories if they don't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Sample name
SAMPLE_NAME = "xenium_sample_01"  # Update with your sample name
H5_FILE = DATA_DIR / f"{SAMPLE_NAME}.h5ad"  # or .h5

# QC thresholds
MIN_GENES = 10  # Minimum genes per cell
MIN_CELLS = 3   # Minimum cells per gene
MAX_MT_PCT = 20  # Maximum mitochondrial percentage
MIN_COUNTS = 50  # Minimum UMI counts per cell

print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Figures directory: {FIGURES_DIR}")

## 2. Load Xenium Data

Load the Xenium data from h5 file. Xenium data includes:
- Gene expression matrix
- Cell spatial coordinates
- Cell metadata

In [ ]:
# Load data
# If using AnnData h5ad format
try:
    adata = sc.read_h5ad(H5_FILE)
    print(f"Loaded AnnData object from {H5_FILE}")
except:
    # Alternative: Load from Xenium output directory
    # adata = sq.read.visium(path=DATA_DIR / SAMPLE_NAME)
    # Or custom loading from h5
    print(f"Please ensure data is in correct format")
    raise

# Display basic information
print(f"\nDataset shape: {adata.shape}")
print(f"Number of cells: {adata.n_obs}")
print(f"Number of genes: {adata.n_vars}")
print(f"\nAvailable observations metadata:")
print(adata.obs.columns.tolist())
print(f"\nAvailable variable metadata:")
print(adata.var.columns.tolist())

## 3. Initial Quality Control Metrics

In [ ]:
# Calculate QC metrics
# Identify mitochondrial genes (if present)
adata.var['mt'] = adata.var_names.str.startswith('MT-')
adata.var['ribo'] = adata.var_names.str.startswith(('RPS', 'RPL'))

# Calculate QC metrics
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=['mt', 'ribo'],
    percent_top=None,
    log1p=False,
    inplace=True
)

print("QC metrics calculated successfully")
print(f"\nNew observation columns:")
print([col for col in adata.obs.columns if 'total' in col or 'pct' in col or 'n_genes' in col])

In [ ]:
# Visualize QC metrics
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Total counts distribution
sc.pl.violin(adata, 'total_counts', ax=axes[0, 0], show=False)
axes[0, 0].set_title('Total counts per cell')

# Number of genes distribution
sc.pl.violin(adata, 'n_genes_by_counts', ax=axes[0, 1], show=False)
axes[0, 1].set_title('Number of genes per cell')

# Mitochondrial percentage
sc.pl.violin(adata, 'pct_counts_mt', ax=axes[0, 2], show=False)
axes[0, 2].set_title('Mitochondrial percentage')

# Scatter plots
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts', ax=axes[1, 0], show=False)
axes[1, 0].set_title('Counts vs Genes')

sc.pl.scatter(adata, x='total_counts', y='pct_counts_mt', ax=axes[1, 1], show=False)
axes[1, 1].set_title('Counts vs MT%')

sc.pl.scatter(adata, x='n_genes_by_counts', y='pct_counts_mt', ax=axes[1, 2], show=False)
axes[1, 2].set_title('Genes vs MT%')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'qc_metrics_before_filtering.png', dpi=300, bbox_inches='tight')
plt.show()

print("QC plots saved")

## 4. Filtering

In [ ]:
# Print pre-filtering stats
print(f"Before filtering:")
print(f"  Number of cells: {adata.n_obs}")
print(f"  Number of genes: {adata.n_vars}")

# Filter cells
sc.pp.filter_cells(adata, min_genes=MIN_GENES)
sc.pp.filter_cells(adata, min_counts=MIN_COUNTS)

# Filter genes
sc.pp.filter_genes(adata, min_cells=MIN_CELLS)

# Filter by mitochondrial percentage
adata = adata[adata.obs.pct_counts_mt < MAX_MT_PCT, :].copy()

print(f"\nAfter filtering:")
print(f"  Number of cells: {adata.n_obs}")
print(f"  Number of genes: {adata.n_vars}")
print(f"  Cells removed: {adata.n_obs}")
print(f"  Genes removed: {adata.n_vars}")

## 5. Normalization and Log-Transformation

In [ ]:
# Store raw counts
adata.layers['counts'] = adata.X.copy()

# Normalize to 10,000 counts per cell
sc.pp.normalize_total(adata, target_sum=1e4)

# Log-transform
sc.pp.log1p(adata)

# Store normalized data
adata.layers['log1p_norm'] = adata.X.copy()

print("Normalization and log-transformation completed")

## 6. Highly Variable Genes Selection

In [ ]:
# Identify highly variable genes
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
    batch_key=None,  # Use batch_key if you have multiple batches
    flavor='seurat_v3',
    subset=False
)

print(f"Number of highly variable genes: {adata.var.highly_variable.sum()}")

# Plot highly variable genes
sc.pl.highly_variable_genes(adata, save='_hvg.png')
plt.savefig(FIGURES_DIR / 'highly_variable_genes.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Dimensionality Reduction

In [ ]:
# Scale data for PCA
sc.pp.scale(adata, max_value=10)

# PCA
sc.tl.pca(adata, svd_solver='arpack', n_comps=50)

# Plot PCA variance ratio
sc.pl.pca_variance_ratio(adata, log=True, n_pcs=50, save='_variance_ratio.png')
plt.savefig(FIGURES_DIR / 'pca_variance_ratio.png', dpi=300, bbox_inches='tight')
plt.show()

print("PCA completed")

In [ ]:
# Compute neighborhood graph
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)

# UMAP
sc.tl.umap(adata)

# Plot UMAP
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sc.pl.umap(adata, color='total_counts', ax=axes[0], show=False, cmap='viridis')
sc.pl.umap(adata, color='n_genes_by_counts', ax=axes[1], show=False, cmap='viridis')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'umap_qc_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

print("UMAP completed")

## 8. Clustering (Initial)

In [ ]:
# Leiden clustering
sc.tl.leiden(adata, resolution=0.5, key_added='leiden_r0.5')
sc.tl.leiden(adata, resolution=1.0, key_added='leiden_r1.0')

# Plot clusters
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sc.pl.umap(adata, color='leiden_r0.5', ax=axes[0], show=False, legend_loc='on data')
sc.pl.umap(adata, color='leiden_r1.0', ax=axes[1], show=False, legend_loc='on data')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'umap_leiden_clusters.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Leiden clustering (r=0.5): {adata.obs['leiden_r0.5'].nunique()} clusters")
print(f"Leiden clustering (r=1.0): {adata.obs['leiden_r1.0'].nunique()} clusters")

## 9. Spatial Visualization

In [ ]:
# Spatial plots require spatial coordinates in adata.obsm['spatial']
# Assuming coordinates are available

if 'spatial' in adata.obsm:
    # Plot spatial distribution colored by clusters
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    sq.pl.spatial_scatter(
        adata,
        color='leiden_r0.5',
        size=1.5,
        ax=axes[0],
        title='Spatial distribution - Leiden (r=0.5)'
    )
    
    sq.pl.spatial_scatter(
        adata,
        color='total_counts',
        size=1.5,
        ax=axes[1],
        cmap='viridis',
        title='Spatial distribution - Total counts'
    )
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'spatial_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("Spatial plots created")
else:
    print("Spatial coordinates not found in adata.obsm['spatial']")
    print("Please add spatial coordinates for spatial visualization")

## 10. Save Preprocessed Data

In [ ]:
# Save preprocessed data
output_file = OUTPUT_DIR / f"{SAMPLE_NAME}_preprocessed.h5ad"
adata.write_h5ad(output_file)

print(f"\nPreprocessed data saved to: {output_file}")
print(f"Final dataset shape: {adata.shape}")
print(f"\nLayers available:")
print(list(adata.layers.keys()))
print(f"\nObsm keys (embeddings):")
print(list(adata.obsm.keys()))

## 11. Summary Statistics

In [ ]:
# Generate summary report
summary = {
    'Sample': SAMPLE_NAME,
    'Total cells': adata.n_obs,
    'Total genes': adata.n_vars,
    'Highly variable genes': adata.var.highly_variable.sum(),
    'Mean counts per cell': adata.obs['total_counts'].mean(),
    'Median counts per cell': adata.obs['total_counts'].median(),
    'Mean genes per cell': adata.obs['n_genes_by_counts'].mean(),
    'Median genes per cell': adata.obs['n_genes_by_counts'].median(),
    'Mean MT%': adata.obs['pct_counts_mt'].mean(),
    'Number of clusters (r=0.5)': adata.obs['leiden_r0.5'].nunique(),
    'Number of clusters (r=1.0)': adata.obs['leiden_r1.0'].nunique(),
}

summary_df = pd.DataFrame([summary]).T
summary_df.columns = ['Value']

print("\n=== Preprocessing Summary ===")
print(summary_df)

# Save summary
summary_df.to_csv(OUTPUT_DIR / f"{SAMPLE_NAME}_preprocessing_summary.csv")
print(f"\nSummary saved to: {OUTPUT_DIR / f'{SAMPLE_NAME}_preprocessing_summary.csv'}")

---

## Next Steps

The preprocessed data is ready for:
1. **Cell type annotation** (02_phenotyping.ipynb)
2. **Spatial analysis** (03_spatial_analysis.ipynb)
3. **Group comparisons** (04_group_comparisons.ipynb)
4. **Integration with Phenocycler data** (06_xenium_phenocycler_integration.ipynb)